# 3D Solid Mechanics

* OpenCascade Technology geometry kernel
* Netgen mesh generator
* NGSolve finite element library

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw 

## CAD geometry

A block with three holes, built from primitives by boolean operations:

In [ ]:
box = Box((0,0,0), (3,0.6,1))
box.faces.name="outer"
cyl = sum( [Cylinder((0.5+i,0,0.5), Y, 0.25,0.8) for i in range(3)] )
cyl.faces.name="cyl"
geo = box-cyl

ea = { "euler_angles" : [-70,5,30] }
Draw(geo, **ea);

find edges between box and cylinders, and build chamfers:

In [ ]:
cylboxedges = geo.faces["outer"].edges * geo.faces["cyl"].edges
cylboxedges.name = "cylbox"
geo = geo.MakeChamfer(cylboxedges, 0.03)

name faces for boundary conditions:

In [ ]:
geo.faces.Min(X).name = "fix"
geo.faces.Max(X).name = "force"

Draw(geo, **ea);

curved tetrahedral mesh of third order:

In [ ]:
mesh = geo.GenerateMesh(maxh=0.1).Curve(3)
Draw (mesh, **ea);

## Linear elasticity

For the displacement $u : \Omega \rightarrow {\mathbb R}^3$ we have the linear strain and the stress by Hooke's law

$$
\varepsilon(u) = \tfrac{1}{2} ( \nabla u + (\nabla u)^T ), \qquad
\sigma = 2 \mu \varepsilon + \lambda \operatorname{tr} \varepsilon \, I,
$$

and the equilibrium of forces $-\operatorname{div} \sigma = f$, with $u = 0$ on $\Gamma_{\text{fix}}$ and traction $\sigma n = g$ on $\Gamma_{\text{force}}$.
The variational formulation: find $u \in H^1(\Omega)^3$ with $u = 0$ on $\Gamma_{\text{fix}}$ such that

$$
\int_\Omega \sigma(\varepsilon(u)) : \varepsilon(v) \, dx = \int_{\Gamma_{\text{force}}} g \cdot v \, ds \qquad \forall \, v.
$$

In [ ]:
E, nu = 210, 0.2
mu  = E / 2 / (1+nu)
lam = E * nu / ((1+nu)*(1-2*nu))

def Stress(strain):
    return 2*mu*strain + lam*Trace(strain)*Id(3)    

Solve with conjugate gradients, preconditioned by BDDC (balancing domain decomposition by constraints):

In [ ]:
fes = VectorH1(mesh, order=3)
u,v = fes.TnT()
lhs = InnerProduct(Stress(Sym(Grad(u))), Sym(Grad(v)))*dx

force = (1e-3,0,0)
rhs = force*v*ds("force")

with TaskManager():
    gfu = Solve(lhs==rhs, u[BND('fix')]==(0,0,0),
                preconditioners.BDDC(),
                solvers.CGSolver.Creator(plotrates=True, tol=1e-8))
print ("ndof =", fes.ndof)

Interpolate the stress into a continuous, matrix-valued finite element space for visualization:

In [ ]:
with TaskManager():
    fesstress = MatrixValued(H1(mesh,order=3), symmetric=True)
    gfstress = GridFunction(fesstress)
    gfstress.Interpolate (Stress(Sym(Grad(gfu))))

In [ ]:
Draw (gfu, mesh, deformation=True, scale=3e4, draw_vol=False, **ea);

In [ ]:
Draw (Norm(gfstress), mesh, deformation=1e4*gfu, draw_vol=False, order=3, **ea);